In [2]:
import numpy as np
import torch
from torch import nn

### 1D - spinless

**Formally:**
Let $\Lambda=\mathbb{Z}/L\mathbb{Z}$ be the 1D lattice with periodic boundary conditions.
Assume $N\le L$. Let $\psi:\Lambda^N\to\mathbb{F}$ ($\mathbb{F}=\mathbb{R}$ or $\mathbb{C}$)
be antisymmetric. Define the set of distinct-site configurations $\Omega:=\{(x_1,\dots,x_N)\in\Lambda^N:\ x_i\neq x_j \text{ in }\Lambda \text{ for all } i\neq j\}.$
Let $z_x:=e^{2\pi i x/L}\in\mathbb{C},\quad x\in\Lambda,$
and define
$F(x_1,\dots,x_N)
:=\prod_{1\le i<j\le N}\bigl(z_{x_j}-z_{x_i}\bigr).$
Then $F$ is antisymmetric and nonzero on $\Omega$, and there exists a symmetric function
$g:\Lambda^N\to\mathbb{F}$ such that
\begin{equation}
\psi(x_1,\dots,x_N)=F(x_1,\dots,x_N)g(x_1,\dots,x_N),
\qquad \forall (x_1,\dots,x_N)\in\Lambda^N.
\end{equation}

**In other words:**
$$\Lambda = \{0, 1, \dots L-1\}$$

In [33]:
class psi_1D_spinless(nn.Module):
    def __init__(
        self,
        L: int,
        hidden_layers: int,
        dim_feedforward: int,
        activation: nn.Module = nn.GELU,
        device=torch.device("cpu"),
    ):
        super().__init__()
        self.L = L
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nn.Parameter(torch.randn(L, device=device))
        # the MLP
        net = [
            nn.Linear(1, dim_feedforward),
            activation(),
        ]
        for _ in range(hidden_layers):
            net.append(nn.Linear(dim_feedforward, dim_feedforward))
            net.append(activation())
        net.append(nn.Linear(dim_feedforward, 2))
        self.g_prime = nn.Sequential(*net)

        self.to(device)

    # def _initialize_weights(m: nn.Module):
    #     if isinstance(m, nn.Linear):

    def F_antisymmetric(self, x: torch.Tensor):
        # x should be of shape (batch_size, n) with entries from \Lambda
        # convert to complex numbers z_x:=e^{2\pi i x/L}
        z_x = torch.exp((2 * torch.pi * 1j * x) / self.L)
        vandermonde_matrix = torch.linalg.vander(
            z_x
        )  # should be of shape (batch_size, n, n)
        # take the determinant
        sign, logabsdet = torch.linalg.slogdet(vandermonde_matrix)
        F = sign * torch.exp(logabsdet)
        return F  # should be a vector of shape (batch_size)

    def eta_symmetric(self, x: torch.Tensor):
        # x should be of shape (batch_size, n) with entries from \Lambda
        N_s = (
            nn.functional.one_hot(x % self.L, num_classes=self.L).sum(dim=1).float()
        )  # (batch_size, L)
        # take matrix-vector product with w_s
        eta = torch.matmul(N_s, self.w_s)
        return eta  # should be a vector of shape (batch_size)

    def forward(self, x: torch.Tensor):
        # x should be of shape (batch_size, n) with entries from \Lambda
        # compute the antisymmetric function F
        F = self.F_antisymmetric(x)  # shape (batch_size), dtype=complex
        # compute the symmetric function g
        # first calculate eta
        eta = self.eta_symmetric(x)  # shape (batch_size)
        # now pass through the g' NN
        g = self.g_prime(eta.unsqueeze(-1))  # shape (batch_size, 2)
        g = torch.complex(g[..., 0], g[..., 1])  # shape (batch_size), dtype=complex
        # return the product of the two
        psi = F * g  # shape (batch_size), dtype=complex
        return psi